In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON


# YAGO 4.5-10

In [2]:
sparql = SPARQLWrapper(
    "https://api.triplydb.com/datasets/mpii/yago/sparql"
)
sparql.setReturnFormat(JSON)

In [24]:
def s_query_given_offset(offset,n):
    q = f'''
    PREFIX yago: <http://yago-knowledge.org/resource/>
    PREFIX schema: <http://schema.org/>
    SELECT ?ent (COUNT(?o) as ?num_triples)
        {{
          select ?ent ?o WHERE {{
            ?ent ?p ?o .
            FILTER(STRSTARTS(STR(?p), STR(yago:)) || STRSTARTS(STR(?p), STR(schema:)))
            FILTER(!isLiteral(?o))
          }} order by ?ent limit {offset} offset {n*offset}
        }} group by ?ent
    '''
    return q


In [25]:
res_dict=dict()
for i in range(10):
    sparql.setQuery(s_query_given_offset(50000,i))
    try:
        ret = sparql.queryAndConvert()

        for r in ret["results"]["bindings"]:
            if r["ent"]['value'] in res_dict.keys():
                res_dict[r["ent"]['value']] += int(r["num_triples"]['value'])
            else:
                res_dict[r["ent"]['value']] = int(r["num_triples"]['value'])
    except Exception as e:
        print(e)

HTTP Error 504: Query has timed out.


KeyboardInterrupt: 

In [20]:
res_dict

{'http://yago-knowledge.org/resource/0-8-4': 3,
 'http://yago-knowledge.org/resource/001_Edizioni_Q30888491': 2,
 'http://yago-knowledge.org/resource/1585_papal_conclave': 44,
 'http://yago-knowledge.org/resource/007_Legends': 2,
 'http://yago-knowledge.org/resource/158Th_Olympiad_Q57337865': 2,
 'http://yago-knowledge.org/resource/1874_United_Kingdom_general_election': 2,
 'http://yago-knowledge.org/resource/007_Racing': 2,
 'http://yago-knowledge.org/resource/158th_New_York_State_Legislature': 3,
 'http://yago-knowledge.org/resource/1874_United_Kingdom_general_election_in_Ireland': 2,
 'http://yago-knowledge.org/resource/18_U003A_12_U003A_00_Q95098566': 1,
 'http://yago-knowledge.org/resource/01-99_Q13050550': 11,
 'http://yago-knowledge.org/resource/1591_papal_conclave': 59,
 'http://yago-knowledge.org/resource/1874_United_States_House_of_Representatives_elections_in_Florida': 3,
 'http://yago-knowledge.org/resource/18_Vendémiaire_Q21606667': 1,
 'http://yago-knowledge.org/resource/

# NELL995 splits

In [4]:
from pykeen.triples import TriplesFactory
import pandas as pd
from rdflib import graph, Namespace, URIRef
from rdflib.namespace import RDF

In [32]:
default_ns = 'https://ste-lod-crew.fr/nell/ontology/'
ns = Namespace(default_ns)
g = graph.Graph()

with open('NELL995/NELLKG0.txt') as inFile:
    with open('NELL995/NELLKG0_with_IRI.txt', 'w') as outFile:
        for line in inFile:
            line=  line.replace('__','_')
            s,p,o = line.split()
            sClass, sName = s.split('_',1)
            oClass, oName = o.split('_',1)

            sClass = default_ns+sClass
            sName = default_ns+sName
            oClass = default_ns+oClass
            oName = default_ns+oName
            pName = default_ns+p

            outFile.write(sClass+'_'+sName + '\t' + pName + '\t' + oClass+'_'+oName + '\n')
            g.add((URIRef(sName), RDF.type, URIRef(sClass)))
            g.add((URIRef(oName), RDF.type, URIRef(oClass)))
            g.add((URIRef(sName), URIRef(pName), URIRef(oName)))

    g.serialize('../datasets/NELL995/NELLKG0_with_IRI.ttl', format='turtle')



In [34]:
tf = TriplesFactory.from_path('NELL995/NELLKG0_with_IRI.txt')
training, testing, validation = tf.split([.8, .1, .1],random_state=42)

In [35]:
pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('NELL995/NELL995_train.txt', header=False, index=False, sep ='\t')
pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('NELL995/NELL995_test.txt', header=False, index=False, sep ='\t')
pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('NELL995/NELL995_valid.txt', header=False, index=False, sep ='\t')
# pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_train.txt', header=False, index=False, sep = '\t')
# pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_test.txt', header=False, index=False, sep = '\t')
# pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('../datasets/NELL995/NELL995_with_IRI_valid.txt', header=False, index=False, sep = '\t')
